In [ ]:
# 1번 셀: 설치 (v3.2-ep04 — 3편과 동일 파이프라인)
EPISODE = "04"
!pip install -q chatterbox-tts requests
!pip uninstall -y -q torchvision
print("설치 완료")

In [ ]:
# 2번 셀: ref.wav 확보 (3편과 같은 목소리 — haesollo-ref 데이터셋)
import glob, shutil, os
found = glob.glob("/kaggle/input/**/ref.wav", recursive=True)
assert found, "ref.wav 없음 — 오른쪽 + Add Input 에서 haesollo-ref 를 붙이세요(3편과 같은 목소리 필수)"
shutil.copyfile(found[0], "ref.wav")
print("ref.wav 확보:", found[0], os.path.getsize("ref.wav"), "bytes")

In [ ]:
# 3번 셀: 대본 내장본 저장 + gen.py 취득(GitHub raw) + 대본 취득부만 로컬 읽기로 치환
# 4편 대본 04.json 은 아직 GitHub main 에 없어 raw 가 404 -> 이 셀에 내장한다.
# 생산 로직(모델·언어·ref·환각 감시선)은 video/kaggle_gen.py 원본 그대로 = 3편과 동일 설정.
import json, hashlib, urllib.request

SCRIPT_JSON = r"""
{
 "episode": "04",
 "title": "1995년 Netscape 상장, 주문 폭주로 첫 거래가 두 시간 막혔다 — 시급 6.85달러 알바생이 만든 브라우저 Mosaic (닷컴 열풍의 방아쇠)",
 "segments": [
  {
   "id": 0,
   "scene": "nasdaq_delay",
   "text": "1995년 8월 9일 아침, 미국 나스닥의 한 종목이 두 시간 가까이 첫 거래를 열지 못했습니다. 사겠다는 주문이 너무 몰려서요. 창업 열여섯 달, 아직 한 푼도 못 번 적자 회사의 주식이었습니다."
  },
  {
   "id": 1,
   "scene": "recap_free_web",
   "text": "지난 편에서, 세른이 웹 기술을 공짜로 풀어버렸다고 했죠. 그런데 그렇게 활짝 열린 웹을, 정작 보통 사람들은 잘 쓰지 않았습니다. 이유는 단순했어요."
  },
  {
   "id": 2,
   "scene": "text_only_web",
   "text": "당시 웹 화면에는 글자밖에 없었습니다. 사진을 보려면 파일을 따로 내려받아, 다른 프로그램으로 열어야 했죠. 학자한테는 충분했지만, 보통 사람한테는 아니었습니다."
  },
  {
   "id": 3,
   "scene": "ncsa_parttimer",
   "text": "무대는 일리노이 대학의 슈퍼컴퓨터 연구소, 엔씨에스에이. 시간당 6달러 85센트짜리 학부생 알바가 있었어요. 동료가 말합니다. 연구 문서인데 글자면 충분하잖아. 알바생이 답하죠. 사람들은 그림 없으면 안 봐요. 대화는 각색, 시급은 실제 기록입니다."
  },
  {
   "id": 4,
   "scene": "img_tag_mail",
   "text": "이 알바생 이름이 마크 앤드리슨입니다. 1993년 2월 25일, 그는 웹 개발자 공개 게시판에 글을 올려요. 새 태그를 제안합니다, 아이엠지. 문서 안에 그림을 바로 박아 넣는 명령이었죠."
  },
  {
   "id": 5,
   "scene": "mosaic_release",
   "text": "1993년 4월 22일, 앤드리슨과 연구소 선배 에릭 비나가 만든 브라우저가 정식 공개됩니다. 이름은 모자이크. 글과 그림이 한 화면에 뜨고, 설치는 몇 번만 누르면 끝이었어요."
  },
  {
   "id": 6,
   "scene": "explosion",
   "text": "반응은 폭발적이었습니다. 웹이 연구자의 도구에서, 누구나 구경하는 놀이터로 바뀐 거죠. 그런데 문제가 생깁니다. 모자이크의 권리는 대학에 있었고, 연구소는 이 폭증을 감당할 조직이 아니었어요."
  },
  {
   "id": 7,
   "scene": "clark_email",
   "text": "1994년 2월, 스물두 살 앤드리슨에게 편지 한 통이 옵니다. 저를 모르시겠지만, 저는 실리콘 그래픽스의 창업자이자 전 회장입니다. 자기 회사를 막 떠난 사업가, 짐 클라크였어요. 이 첫 줄은 기록에 남은 실제 문장입니다."
  },
  {
   "id": 8,
   "scene": "rewrite_mozilla",
   "text": "둘은 1994년 4월 회사를 세웁니다. 그런데 모자이크 코드는 한 줄도 못 씁니다. 권리가 대학에 있으니까요. 그래서 처음부터 새로 만들어요. 사내 코드명은 모질라, 모자이크를 잡아먹겠다는 뜻이었습니다."
  },
  {
   "id": 9,
   "scene": "netscape_wins",
   "text": "넷스케이프로 이름을 바꾼 이들의 브라우저는, 나오자마자 원조를 밀어냅니다. 1995년 점유율은 조사마다 다르지만 80퍼센트 안팎. 세상이 웹을 내다보는 창문이, 사실상 하나였습니다."
  },
  {
   "id": 10,
   "scene": "ipo_day",
   "text": "그리고 1995년 8월 9일. 넷스케이프가 주식 시장에 나옵니다. 공모가는 한 주에 28달러. 그런데 사겠다는 주문이 물량을 압도해서, 이 주식의 첫 거래를 두 시간 가까이 열지 못했습니다."
  },
  {
   "id": 11,
   "scene": "ipo_numbers",
   "text": "겨우 열린 첫 거래 가격은 71달러. 장중 최고 74달러 75센트까지 올랐다가, 58달러 25센트로 마감합니다. 창업 열여섯 달, 이익은 아직 없는 회사의 몸값이 하루 만에 30억 달러 가까이 됐어요."
  },
  {
   "id": 12,
   "scene": "dotcom_trigger",
   "text": "이날은 닷컴 열풍의 방아쇠로 불립니다. 얼마를 벌었느냐가 아니라, 인터넷이라는 단어가 붙었느냐로 값이 매겨지는 시대죠. 오늘의 인공지능 투자 열기와 겹쳐 보인다면, 그건 제 관찰입니다."
  },
  {
   "id": 13,
   "scene": "law_giant",
   "text": "여기서 이 시리즈의 법칙. 모든 해결은 새로운 문제를 낳는다. 이 요란한 성공이 거인을 깨웁니다. 같은 달, 마이크로소프트가 인터넷 익스플로러를 세상에 내놓거든요. 그리고 숙제는 하나 더 있었습니다."
  },
  {
   "id": 14,
   "scene": "next",
   "text": "그림은 떴지만, 웹은 여전히 눌러도 반응 없는 전단지였어요. 여러분이라면 이 문제를 며칠 만에 풀 수 있을까요? 댓글로 알려주세요. 다음 편은, 넷스케이프 개발자 한 명이 열흘 만에 만든 언어, 자바스크립트 이야기입니다. 구독과 좋아요 눌러두시면, 다음 편에서 뵙겠습니다."
  }
 ]
}
"""

with open(f"script_{EPISODE}.json", "w", encoding="utf-8") as f:
    f.write(SCRIPT_JSON.strip())
s = json.load(open(f"script_{EPISODE}.json", encoding="utf-8"))
assert s["episode"] == "04" and len(s["segments"]) == 15
print("대본 md5:", hashlib.md5(json.dumps(s, ensure_ascii=False, indent=1).encode()).hexdigest())

URL = "https://raw.githubusercontent.com/nous-zero/tech-history/main/video/kaggle_gen.py"
g = urllib.request.urlopen(URL).read().decode("utf-8")
out, patched = [], 0
NL = chr(10)
for ln in g.split(NL):
    if ln.startswith('url = f"https://raw'):
        patched += 1
        continue
    if ln.startswith("script = requests.get"):
        ln = 'script = json.load(open(f"script_{EPISODE}.json", encoding="utf-8"))'
        patched += 1
    out.append(ln)
gen = NL.join(out)
assert patched == 2 and "requests.get" not in gen, (patched, "requests.get" in gen)
with open("gen.py", "w", encoding="utf-8") as f:
    f.write(gen)
compile(gen, "gen.py", "exec")
print("[v3.2-ep04] gen.py 준비 완료 — 원본 kaggle_gen.py md5:", hashlib.md5(g.encode()).hexdigest())

In [ ]:
# 4번 셀: 음성 생산 — 새 파이썬 프로세스(서브프로세스 격리)
ONLY = []   # 빈 목록 = 15조각 전부

import json, sys
json.dump({"episode": EPISODE, "only": ONLY}, open("gen_config.json", "w"))
!{sys.executable} -u gen.py

In [ ]:
# 5번 셀: 산출 검증 + 압축
import glob, os, shutil, hashlib, wave, contextlib
wavs = sorted(glob.glob(f"voice_{EPISODE}_fix/seg*.wav"))
print("생산된 조각:", len(wavs), "개")
want = len(ONLY) if ONLY else 15   # 부분 재생산이면 ONLY 개수가 기대치다(전량 15 고정 가드는 오탐)
assert len(wavs) == want, f"기대 {want}개, 실제 {len(wavs)}개"
tot = 0.0
for w in wavs:
    with contextlib.closing(wave.open(w)) as f:
        d = f.getnframes() / f.getframerate()
    tot += d
    print(os.path.basename(w), round(d, 2), "s", hashlib.md5(open(w, "rb").read()).hexdigest()[:8])
print("합산", round(tot, 2), "s")
zip_path = shutil.make_archive(f"voice_{EPISODE}_fix", "zip", f"voice_{EPISODE}_fix")
print("DONE", zip_path, os.path.getsize(zip_path), "bytes")